# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring.**

Four notebooks built to this point. ML-04 contracted the data, ML-05 froze and audited the feature
vector, ML-06 audited the signals, ML-07 wrote the rule this model has to beat. The order matters:
there is now something honest to beat, and clean features to beat it with.

| Card asks for | Lives in |
|---|---|
| Method choice and why | §1 |
| Split design — grouped? time-aware? why honest | §2 |
| Train + compare vs the Week-4 baseline, same data / metric / split | §3 |
| Errors and interpretation | §4 |

**The one change that makes this week different.** Every notebook so far ranked by `ctr_gap_pp` — an
**authored proxy** measured in the same month as the features. Nothing observed ever confirmed it.
This week the target becomes a **forward-looking outcome**: features from March, outcome from
**April**. For the first time the label is something the world did, not something I defined.

## 0. Setup — March features, April outcome

Two partitions. March is the observation window (identical to ML-04/05/06/07); April supplies the
outcome and is never used to build a feature.

In [41]:
%pip -q install --upgrade duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [42]:
import duckdb, pandas as pd, numpy as np, sklearn, scipy
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
SEED = 42
print(f"versions -> duckdb {duckdb.__version__} | sklearn {sklearn.__version__} | "
      f"pandas {pd.__version__} | numpy {np.__version__}")
print("(tree-ensemble numbers can move a little between library versions -- seeds fixed at 42, "
      "n_jobs=1 everywhere so a rerun reproduces the table exactly)")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
def month_rel(m): return f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"
DIMC = f"read_parquet('{REL}/dim_content.parquet')"

FEATURE_MONTH, OUTCOME_MONTH = "2026-03", "2026-04"
DECISION_MOMENT = "2026-04-01"
MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0   # ML-04 eligibility, unchanged
GAP_MIN, CLICKS_MIN = 0.10, 10                                  # ML-07 thresholds, unchanged
OUT_MIN_IMPRESSIONS = 100                                       # April floor, lighter (see 2)

versions -> duckdb 1.5.5 | sklearn 1.6.1 | pandas 2.2.2 | numpy 2.0.2
(tree-ensemble numbers can move a little between library versions -- seeds fixed at 42, n_jobs=1 everywhere so a rerun reproduces the table exactly)


In [43]:
def page_month(month):
    """One row per page for one month. Identical aggregation to ML-04/05/06/07."""
    df = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                                AS impressions,
               SUM(gsc_clicks)                                                     AS clicks,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)  AS active_days,
               MAX(gsc_impressions)                                                AS top_day_impressions,
               SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_num,
               SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_den,
               STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS position_volatility,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{month}-18') AS imp_last14,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{month}-04'
                                              AND report_date <  DATE '{month}-18') AS imp_prev14
        FROM {month_rel(month)}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    """).df()
    # +1: gsc_avg_position is zero-based (proved in ML-04 3.4)
    df["avg_position"] = df.pos_num / df.pos_den.replace(0, np.nan) + 1
    df["ctr_pp"]       = 100 * df.clicks / df.impressions.replace(0, np.nan)
    return df

mar = page_month(FEATURE_MONTH)
apr = page_month(OUTCOME_MONTH)
print(f"{FEATURE_MONTH}: {len(mar):,} pages | {OUTCOME_MONTH}: {len(apr):,} pages")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03: 176,738 pages | 2026-04: 194,760 pages


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### First: what am I actually predicting now?

Through ML-07 the score answered *"is this page below its position peers **right now**?"* — an
authored proxy, computed in the same window as the features. A model trained on that would be
learning my own arithmetic back.

The editor's real question is different, and it is about the future:

> *"If I spend an hour on this page, is the problem still going to be there — or would it have
> sorted itself out anyway?"*

So the label is **persistence**, observed one month later:

```text
label = 1  if the page would STILL be flagged in April
         (April gap >= 0.10pp AND April missed clicks >= 10 -- the exact ML-07 thresholds)
label = 0  otherwise
```

Same rule, next month, applied to what actually happened. **This is an observed outcome, not a
rule I authored** — the first one in the project. A page that fixes itself is a waste of an
editor's hour, and the March queue currently cannot tell those apart.

### The important consequence: `clicks_31d` is no longer leakage

ML-07 ended with a rule I wrote for myself: *the model must beat this baseline without using
clicks.* That constraint came from the target being arithmetic on clicks in the same window.

**Moving the label to April dissolves it.** Leakage is not a property of a column — it is a
relationship between a column and a label. March clicks are now simply history, fully knowable at
the decision moment, and strictly before the outcome window. Forbidding them would be superstition.

Rather than assume that, I test it: one model gets **shape features only**, another gets shape
features **plus the March gap**. If prior performance is all that matters, the shape-only model
loses and I say so.

### Which methods

| Question shape | Method | Why |
|---|---|---|
| yes/no with an observed label | **Logistic Regression** first | readable coefficients; if a linear model gets most of the way, complexity is not earning its keep |
| ...then a stronger learner | **Random Forest** | handles the non-linear, heavy-tailed shapes ML-06 measured |
| "which first?" ranking | classifier **probability**, scored at **precision@K** | an editor works a queue top-down; the score is a ranking device, not a verdict |
| "what does it lean on?" | **permutation importance** | importance from a fit, checked by shuffling |

Deliberately not used: gradient boosting. On this data it would likely gain a point or two and cost
the ability to explain a rank to a reviewer. The skill's line is the right one — a model two points
stronger that nobody can read is not two points better. If the comparison had shown the simple
models failing, that would be the argument for adding complexity; §3 reports whether it does.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Two independent honesty problems, two separate defences.

**Time — solved by construction, not by the split.** Every feature is aggregated from
`report_date <= 2026-03-31`; every outcome from April. The prediction moment sits between them.
No feature window overlaps the label window, so there is no future information to leak.

```text
   FEATURES: March 2026                  OUTCOME: April 2026            June 2026
   |-----------------------------|      |----------------------|        [SEALED]
   03-01                     03-31      04-01              04-30
                                   |
                          decision moment 2026-04-01
```

**Groups — solved by the split.** ML-06 signal 1 measured pooled CTR running from 0.115% to 1.159%
across clients, a **10x spread**. Rows from one client share a hidden character that heavily
determines the answer, so a random split lets a model memorise the client and report skill it does
not have. Validation is therefore **GroupKFold(5) on `client_hash_id`** — every metric is
out-of-fold, and every test page belongs to a client the model never trained on.

I report the random-split number too. Per the leakage skill, **the gap between them is itself a
finding** about how much memorisation was available.

**Population, and the one honest compromise.** A page needs April data for a label to exist. Pages
that vanish in April are dropped — and that is selection using outcome-window information. It is a
choice, not a crime, but hiding it would be, so §2's code counts them and §4 states the limit. The
April floor is lighter than March's (100 impressions, not 500) because it only has to make the
outcome *readable*, not make a page worth an editor's time.

In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Build the March feature frame (ML-04 eligibility, ML-05 feature vector) ---
lane = mar[(mar.impressions >= MIN_IMPRESSIONS) &
           (mar.active_days >= MIN_ACTIVE_DAYS) &
           (mar.avg_position >= MIN_POSITION)].copy()
print(f"March eligible lane: {len(lane):,} pages, {lane.client_hash_id.nunique()} clients "
      f"(matches ML-04/05/06/07)")

# ML-05's vector, rebuilt
lane["log_impressions_31d"]      = np.log1p(lane.impressions)
lane["days_with_impressions_31d"] = lane.active_days
lane["position_volatility_31d"]  = lane.position_volatility
lane["top_day_impression_share"] = lane.top_day_impressions / lane.impressions
lane["momentum_log14v14"]        = np.log((lane.imp_last14.fillna(0) + 1) /
                                          (lane.imp_prev14.fillna(0) + 1))
lane["impressions_share_of_client"] = (lane.impressions /
                                       lane.groupby("client_hash_id").impressions.transform("sum"))

dimc = con.sql(f"SELECT content_hash_id, word_count, content_type FROM {DIMC}").df()
lane = lane.merge(dimc, on="content_hash_id", how="left", validate="many_to_one")
wc = pd.to_numeric(lane.word_count, errors="coerce").astype("float64")
lane["content_meta_missing"] = wc.isna().astype(int)
lane["log_word_count"] = np.log1p(wc.fillna(wc.groupby(lane.client_hash_id).transform("median"))
                                    .fillna(wc.median()))
cat_src  = lane.content_type.astype("object")
top_cats = cat_src.value_counts().head(6).index
lane["cat_clean"] = cat_src.where(cat_src.isin(top_cats), "other").fillna("missing")
dummies = pd.get_dummies(lane.cat_clean, prefix="cat", drop_first=True).astype(int)
lane = pd.concat([lane, dummies], axis=1)

SHAPE_FEATURES = ["log_impressions_31d", "days_with_impressions_31d", "position_volatility_31d",
                  "top_day_impression_share", "momentum_log14v14", "impressions_share_of_client",
                  "log_word_count", "content_meta_missing"] + list(dummies.columns)
print(f"ML-05 shape features rebuilt: {len(SHAPE_FEATURES)}")

March eligible lane: 61,881 pages, 36 clients (matches ML-04/05/06/07)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML-05 shape features rebuilt: 10


In [46]:
# --- March gap (the baseline's own signal) + the April outcome ---------------
def add_gap(df, tag):
    """Leave-one-out, volume-weighted peer baseline within position tier. Same as ML-04."""
    df = df.copy()
    df["tier"] = df.avg_position.apply(
        lambda p: "top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
        else "page_3_5" if p <= 50 else "deep")
    tc = df.groupby("tier").clicks.transform("sum")
    ti = df.groupby("tier").impressions.transform("sum")
    df[f"{tag}_peer_pp"]      = (100 * (tc - df.clicks) /
                                 (ti - df.impressions)).replace([np.inf, -np.inf], np.nan)
    df[f"{tag}_gap_pp"]       = df[f"{tag}_peer_pp"] - df.ctr_pp
    df[f"{tag}_missed_clicks"] = df.impressions * df[f"{tag}_gap_pp"] / 100
    return df

lane = add_gap(lane, "mar")
lane["mar_ctr_pp"] = lane.ctr_pp
lane["log_clicks_31d"] = np.log1p(lane.clicks)
PRIOR_FEATURES = ["mar_gap_pp", "mar_ctr_pp", "log_clicks_31d"]

# April: eligibility only needs to make the outcome READABLE
apr_e = apr[(apr.impressions >= OUT_MIN_IMPRESSIONS) & (apr.avg_position >= MIN_POSITION)].copy()
apr_e = add_gap(apr_e, "apr")
apr_e["label"] = ((apr_e.apr_gap_pp >= GAP_MIN) & (apr_e.apr_missed_clicks >= CLICKS_MIN)).astype(int)

n_before = len(lane)
df = lane.merge(apr_e[["content_hash_id", "label", "apr_gap_pp", "apr_missed_clicks",
                       "impressions", "avg_position"]]
                .rename(columns={"impressions": "apr_impressions",
                                 "avg_position": "apr_position"}),
                on="content_hash_id", how="inner", validate="one_to_one")
df = df.dropna(subset=SHAPE_FEATURES + PRIOR_FEATURES + ["label"]).copy()

print(f"March eligible pages          : {n_before:,}")
print(f"...also present & readable in April: {len(df):,} ({len(df)/n_before:.1%})")
print(f"...DROPPED for no April outcome    : {n_before - len(df):,} "
      f"({(n_before-len(df))/n_before:.1%})  <- disclosed selection, see section 4")
print(f"\nBASE RATE: {df.label.mean():.1%} of pages would still be flagged in April "
      f"({int(df.label.sum()):,} of {len(df):,})")
print(f"clients: {df.client_hash_id.nunique()}")

March eligible pages          : 61,881
...also present & readable in April: 60,942 (98.5%)
...DROPPED for no April outcome    : 939 (1.5%)  <- disclosed selection, see section 4

BASE RATE: 9.5% of pages would still be flagged in April (5,786 of 60,942)
clients: 34


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same rows, same split, same metrics for every line of the table.** The baseline is not quoted
from ML-07 — it is recomputed here on these rows, so the comparison is like for like.

**The metrics, and why these.**

- **precision@50 / @200** — the decision is a queue an editor works top-down. Precision@K asks the
  only question that matters at the top of the list: *of the first K pages I open, how many are
  real, persistent problems?* K=50 is about a week of review; K=200 about a month.
- **ROC AUC** — ranking quality across the whole list, not just the top.
- **The base rate sits next to every number.** Precision@50 of 70% is meaningless until you know
  whether random picking gives 65% or 20%. **Lift** = precision ÷ base rate is the honest summary.

Five lines: random ranking (the floor), the ML-07 rule, then three models of increasing ambition.

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

y      = df.label.values
groups = df.client_hash_id.values
gkf    = GroupKFold(n_splits=5)

for tr, te in gkf.split(df[SHAPE_FEATURES].values, y, groups):
    assert set(groups[tr]).isdisjoint(set(groups[te])), "client appears on both sides"
print(f"split check PASS: no client appears in both train and test in any of "
      f"{gkf.get_n_splits()} folds")

def precision_at_k(scores, y_true, k, tie_seed=SEED):
    """Rank by score; break EXACT ties with a seeded jitter so the file order
    (or alphabetical order, or whatever pandas happened to do) never decides the queue."""
    s = np.asarray(scores, dtype=float)
    jitter = np.random.default_rng(tie_seed).random(len(s)) * 1e-12
    order = np.lexsort((jitter, -s))[:k]
    return float(np.mean(np.asarray(y_true)[order]))

def out_of_fold(make_model, feats):
    """Fit inside each fold, predict the held-out clients. Every number is out-of-fold."""
    oof = np.zeros(len(df))
    X = df[feats].values
    for tr, te in gkf.split(X, y, groups):
        m = make_model().fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return oof

def row(name, scores):
    return {"model": name,
            "precision@50":  round(precision_at_k(scores, y, 50), 3),
            "precision@200": round(precision_at_k(scores, y, 200), 3),
            "ROC AUC":       round(roc_auc_score(y, scores), 3)}

rng = np.random.default_rng(SEED)
results = [
    row("random ranking (base rate)", rng.random(len(df))),
    row("ML-07 rule (March missed clicks)", df.mar_missed_clicks.values),
]

logreg = lambda: make_pipeline(StandardScaler(),
                               LogisticRegression(max_iter=2000, random_state=SEED))
rf     = lambda: RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                                        random_state=SEED, n_jobs=1)

oof_lr        = out_of_fold(logreg, SHAPE_FEATURES)
oof_rf        = out_of_fold(rf,     SHAPE_FEATURES)
oof_rf_prior  = out_of_fold(rf,     SHAPE_FEATURES + PRIOR_FEATURES)

results += [row("LogReg  - shape features only", oof_lr),
            row("RF      - shape features only", oof_rf),
            row("RF      - shape + March gap",   oof_rf_prior)]

table = pd.DataFrame(results)
base = df.label.mean()
table["lift@50"] = (table["precision@50"] / base).round(2)
print(f"BASE RATE (a random queue) = {base:.3f}\n")
display(table)

cut = np.sort(oof_rf_prior)[::-1][49]
print(f"tie policy: score descending, exact ties broken by seeded jitter (seed {SEED}).")
print(f"rows sharing the exact precision@50 cutoff score: {int((oof_rf_prior == cut).sum())}")

split check PASS: no client appears in both train and test in any of 5 folds
BASE RATE (a random queue) = 0.095



,model,precision@50,precision@200,ROC AUC,lift@50
0,random ranking (base rate),0.12,0.085,0.494,1.26
1,ML-07 rule (March missed clicks),0.90,0.875,0.864,9.48
2,LogReg - shape features only,0.48,0.385,0.878,5.06
3,RF - shape features only,0.90,0.845,0.881,9.48
4,RF - shape + March gap,1.00,0.980,0.922,10.53


tie policy: score descending, exact ties broken by seeded jitter (seed 42).
rows sharing the exact precision@50 cutoff score: 1


In [48]:
# --- The same comparison, fold by fold: is the winner a STABLE winner? -------
def per_fold(make_model, feats, K=50):
    rows, X = [], df[feats].values
    for i, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        p = make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        rows.append({"fold": i,
                     "test_clients": df.iloc[te].client_hash_id.nunique(),
                     "test_rows": len(te),
                     "base_rate": round(float(y[te].mean()), 3),
                     "model_p@50": round(precision_at_k(p, y[te], K), 3),
                     "rule_p@50":  round(precision_at_k(df.mar_missed_clicks.values[te], y[te], K), 3)})
    out = pd.DataFrame(rows)
    out["model - rule"] = (out["model_p@50"] - out["rule_p@50"]).round(3)
    return out

folds = per_fold(rf, SHAPE_FEATURES + PRIOR_FEATURES)
display(folds)

wins = int((folds["model - rule"] > 0).sum())
print(f"the model beats the rule in {wins} of {len(folds)} folds "
      f"(mean margin {folds['model - rule'].mean():+.3f}, "
      f"range {folds['model - rule'].min():+.3f} to {folds['model - rule'].max():+.3f})")
print(f"fold base rates run from {folds.base_rate.min():.3f} to {folds.base_rate.max():.3f} -- "
      f"some folds are simply easier queues than others, which is part of the spread.")
print("\nOBSERVED: " + ("the model wins in every fold, so the pooled margin is not hiding a flip."
      if wins == len(folds) else
      f"the pooled table shows one number; the folds show {wins} wins and {len(folds)-wins} losses. "
      "The direction and the wobble get reported together -- an average winner is not "
      "automatically a stable winner."))
print(f"\nfold sizes in CLIENTS: {folds.test_clients.tolist()} -- GroupKFold balances by rows, "
      f"not by clients, and my largest client is {folds.test_rows.max()/len(df):.0%} of the frame, "
      f"so it gets a fold to itself. Fold 1 is therefore ONE client's story rather than an average "
      f"of several, and its base rate ({folds.base_rate.iloc[0]:.3f}) is the highest of the five. "
      f"The five folds are not five equally-informative exams.")

,fold,test_clients,test_rows,base_rate,model_p@50,rule_p@50,model - rule
0,1,1,14171,0.149,1.00,0.90,0.10
1,2,8,11681,0.123,1.00,0.84,0.16
2,3,4,11726,0.067,0.86,0.80,0.06
3,4,10,11683,0.072,0.96,0.92,0.04
4,5,11,11681,0.052,0.88,0.80,0.08


the model beats the rule in 5 of 5 folds (mean margin +0.088, range +0.040 to +0.160)
fold base rates run from 0.052 to 0.149 -- some folds are simply easier queues than others, which is part of the spread.

OBSERVED: the model wins in every fold, so the pooled margin is not hiding a flip.

fold sizes in CLIENTS: [1, 8, 4, 10, 11] -- GroupKFold balances by rows, not by clients, and my largest client is 23% of the frame, so it gets a fold to itself. Fold 1 is therefore ONE client's story rather than an average of several, and its base rate (0.149) is the highest of the five. The five folds are not five equally-informative exams.


In [49]:
# --- Is the gap between the best model and the baseline real, or noise? -----
model_scores = {"2. LogReg  - shape features only": oof_lr,
                "3. RF      - shape features only": oof_rf,
                "4. RF      - shape + March gap":   oof_rf_prior}

# Pick the best model from its own out-of-fold scores, not by string-matching the table --
# a stray space in a label should never be able to break the comparison.
p200 = {name: precision_at_k(s, y, 200) for name, s in model_scores.items()}
best_name   = max(p200, key=p200.get)
best_scores = model_scores[best_name]

base_scores = df.mar_missed_clicks.values
boot = []
idx = np.arange(len(df))
for b in range(500):
    s = rng.choice(idx, size=len(idx), replace=True)
    boot.append(precision_at_k(best_scores[s], y[s], 200) -
                precision_at_k(base_scores[s], y[s], 200))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"best by precision@200: {best_name}")
print(f"difference vs the ML-07 rule at K=200: {np.mean(boot):+.3f} "
      f"(95% bootstrap interval {lo:+.3f} to {hi:+.3f}, 500 resamples)")
print("-> the interval EXCLUDES zero, so the difference is unlikely to be resampling noise."
      if lo > 0 or hi < 0 else
      "-> the interval CONTAINS zero. On this evidence I cannot claim a real difference at K=200; "
      "reporting it as a tie is the honest call.")

best by precision@200: 4. RF      - shape + March gap
difference vs the ML-07 rule at K=200: +0.103 (95% bootstrap interval +0.055 to +0.150, 500 resamples)
-> the interval EXCLUDES zero, so the difference is unlikely to be resampling noise.


In [50]:
# --- The grouped vs random split gap: how much memorisation was on offer? ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
Xs = df[SHAPE_FEATURES].values

tr_g, te_g = next(gss.split(Xs, y, groups))
auc_grouped = roc_auc_score(y[te_g], rf().fit(Xs[tr_g], y[tr_g]).predict_proba(Xs[te_g])[:, 1])

perm = rng.permutation(len(df)); cut = int(0.75 * len(df))
tr_r, te_r = perm[:cut], perm[cut:]
auc_random = roc_auc_score(y[te_r], rf().fit(Xs[tr_r], y[tr_r]).predict_proba(Xs[te_r])[:, 1])

print(f"RF, shape features, single split:")
print(f"  random split  (clients appear in BOTH sides) : ROC AUC {auc_random:.3f}")
print(f"  grouped split (held-out clients only)        : ROC AUC {auc_grouped:.3f}")
print(f"  gap                                          : {auc_random - auc_grouped:+.3f}")
print("\n-> that gap is the memorisation that a random split would have sold me as skill. It is "
      "why every number in the table above is grouped and out-of-fold.")

RF, shape features, single split:
  random split  (clients appear in BOTH sides) : ROC AUC 0.909
  grouped split (held-out clients only)        : ROC AUC 0.893
  gap                                          : +0.016

-> that gap is the memorisation that a random split would have sold me as skill. It is why every number in the table above is grouped and out-of-fold.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

A metric without error analysis is decoration. Three questions: what does the model lean on, where
is it most wrong, and what do the wrong cases look like up close.

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

# Importance measured on HELD-OUT clients, by shuffling one column at a time.
model_full = rf().fit(df[SHAPE_FEATURES + PRIOR_FEATURES].values[tr_g], y[tr_g])
pi = permutation_importance(model_full,
                            df[SHAPE_FEATURES + PRIOR_FEATURES].values[te_g], y[te_g],
                            n_repeats=10, random_state=SEED, n_jobs=1, scoring="roc_auc")
imp = (pd.DataFrame({"feature": SHAPE_FEATURES + PRIOR_FEATURES,
                     "drop_in_auc": pi.importances_mean.round(4),
                     "sd": pi.importances_std.round(4)})
       .sort_values("drop_in_auc", ascending=False).reset_index(drop=True))
display(imp)

top1 = imp.iloc[0]
ratio = imp.iloc[0].drop_in_auc / max(imp.iloc[1].drop_in_auc, 1e-9)
print(f"top feature: {imp.iloc[0].feature}  (shuffling it costs {imp.iloc[0].drop_in_auc:.4f} AUC, "
      f"{ratio:.1f}x the next one)")
if ratio > 2:
    print("INVESTIGATE, do not celebrate. One feature towering over the rest is the leakage symptom "
          "the skill warns about. It is NOT temporal leakage here -- March impressions are knowable "
          "on 2026-04-01. It is the label's own construction showing through: the label requires "
          "April missed clicks >= 10, and missed clicks = impressions x gap / 100, so a large page "
          "clears that gate on almost any positive gap while a 600-impression page structurally "
          "cannot. The next cell tests exactly how much of the score that alone explains.")
else:
    print("Several features contribute at similar magnitude -- the shape of weak signal used "
          "honestly rather than a model that found the answer.")
print(f"\nnear-zero or negative contributors: "
      f"{', '.join(imp[imp.drop_in_auc <= 0.0005].feature.tolist()) or 'none'}")
print("Two checks land here. ML-05 predicted the content_type one-hots would carry nothing (99.3% "
      "of pages share one level) -- confirmed. And impressions_share_of_client, the feature ML-05 "
      "found STRONGEST against the authored proxy, contributes nothing against the observed label. "
      "A feature's worth is a property of the question, not of the feature.")

,feature,drop_in_auc,sd
0,log_impressions_31d,0.1244,0.0012
1,mar_gap_pp,0.0390,0.0008
2,log_clicks_31d,0.0142,0.0008
3,mar_ctr_pp,0.0135,0.0005
4,position_volatility_31d,0.0096,0.0005
5,momentum_log14v14,0.0064,0.0005
6,days_with_impressions_31d,0.0019,0.0003
7,top_day_impression_share,0.0010,0.0003
8,log_word_count,0.0008,0.0002
9,content_meta_missing,0.0004,0.0001


top feature: log_impressions_31d  (shuffling it costs 0.1244 AUC, 3.2x the next one)
INVESTIGATE, do not celebrate. One feature towering over the rest is the leakage symptom the skill warns about. It is NOT temporal leakage here -- March impressions are knowable on 2026-04-01. It is the label's own construction showing through: the label requires April missed clicks >= 10, and missed clicks = impressions x gap / 100, so a large page clears that gate on almost any positive gap while a 600-impression page structurally cannot. The next cell tests exactly how much of the score that alone explains.

near-zero or negative contributors: content_meta_missing, cat_feedly article, cat_keyword article, impressions_share_of_client
Two checks land here. ML-05 predicted the content_type one-hots would carry nothing (99.3% of pages share one level) -- confirmed. And impressions_share_of_client, the feature ML-05 found STRONGEST against the authored proxy, contributes nothing against the observed labe

In [52]:
# --- How much of the score is just "this page is big"? ----------------------
size_only = row("size only: rank by March impressions", df.impressions.values)
gap_only  = row("gap only: rank by March gap (pp)",     df.mar_gap_pp.values)

# Recompute the winning model's row from its scores rather than looking it up by label.
best_row = row("RF - shape + March gap", oof_rf_prior)
print(pd.DataFrame([size_only, gap_only, best_row]).to_string(index=False))

top200_by_size = df.nlargest(200, "impressions")
print(f"\nbase rate overall                     : {base:.3f}")
print(f"base rate among the 200 largest pages : {top200_by_size.label.mean():.3f}")
print("\nOBSERVED, and it cuts against what I expected. Size alone reaches precision@50 of "
      f"{size_only['precision@50']:.2f} and gap alone {gap_only['precision@50']:.2f}, against "
      f"{best_row['precision@50']:.2f} for the model -- so neither ingredient explains the score "
      "on its own, and the model is combining them rather than acting as a size detector.")
print("Two things are true at once. The LABEL is size-influenced by construction: the "
      ">=10-missed-clicks gate takes the base rate from 0.4% in the 500-1k band to 47% at 20k+, "
      f"and size alone reaches ROC AUC {size_only['ROC AUC']:.3f} across the whole ranking. But the "
      "MODEL's top-of-queue precision is not reducible to that -- combining size with the March gap "
      f"is what lifts precision@50 from {max(size_only['precision@50'], gap_only['precision@50']):.2f} "
      f"to {best_row['precision@50']:.2f}. The easy label goes in the limitations; the combination "
      "is a real result.")

                               model  precision@50  precision@200  ROC AUC
size only: rank by March impressions          0.42          0.415    0.875
    gap only: rank by March gap (pp)          0.50          0.330    0.656
              RF - shape + March gap          1.00          0.980    0.922

base rate overall                     : 0.095
base rate among the 200 largest pages : 0.415

OBSERVED, and it cuts against what I expected. Size alone reaches precision@50 of 0.42 and gap alone 0.50, against 1.00 for the model -- so neither ingredient explains the score on its own, and the model is combining them rather than acting as a size detector.
Two things are true at once. The LABEL is size-influenced by construction: the >=10-missed-clicks gate takes the base rate from 0.4% in the 500-1k band to 47% at 20k+, and size alone reaches ROC AUC 0.875 across the whole ranking. But the MODEL's top-of-queue precision is not reducible to that -- combining size with the March gap is what lifts

In [53]:
# --- Where is the model most wrong? -----------------------------------------
df["p"] = oof_rf_prior
df["baseline_rank"] = (-df.mar_missed_clicks).argsort().argsort() + 1
df["model_rank"]    = (-df.p).argsort().argsort() + 1

def auc_or_nan(sub):
    return roc_auc_score(sub.label, sub.p) if sub.label.nunique() > 1 else np.nan

def slice_report(col, bins, labels, title):
    df["_b"] = pd.cut(df[col], bins=bins, labels=labels)
    rows = []
    for name, sub in df.groupby("_b", observed=True):
        rows.append({"band": name, "n_pages": len(sub),
                     "base_rate": round(sub.label.mean(), 3),
                     "model_auc": round(auc_or_nan(sub), 3) if sub.label.nunique() > 1 else np.nan})
    out = pd.DataFrame(rows).set_index("band")
    print(f"\n{title}")
    display(out[out.n_pages >= 50])          # 50-row floor, same as ML-06

slice_report("avg_position", [0, 2, 3, 5, 10, 20, 50, np.inf],
             ["1.0-2", "2-3", "3-5", "5-10", "10-20", "20-50", "50+"],
             "By position band -- does the model work where ML-06 found the anomaly?")
slice_report("impressions", [0, 1000, 5000, 20000, np.inf],
             ["500-1k", "1k-5k", "5k-20k", "20k+"], "By March traffic volume")

rows = []
for cid, sub in df.groupby("client_hash_id"):
    if len(sub) >= 50 and sub.label.nunique() > 1:
        rows.append({"n": len(sub), "base_rate": sub.label.mean(), "auc": auc_or_nan(sub)})
by_client = pd.DataFrame(rows).dropna()
print(f"\nPer-client AUC across {len(by_client)} clients with >= 50 pages: "
      f"worst {by_client.auc.min():.3f} | median {by_client.auc.median():.3f} | "
      f"best {by_client.auc.max():.3f}")
print("-> ML-06 signal 1 measured a 10x CTR spread between clients; this is that spread showing up "
       "as unevenness in where the model can be trusted.")


By position band -- does the model work where ML-06 found the anomaly?


,n_pages,base_rate,model_auc
band,,,
1.0-2,433,0.115,0.916
2-3,2322,0.104,0.911
3-5,11470,0.130,0.921
5-10,23229,0.128,0.919
10-20,11739,0.043,0.898
20-50,11063,0.047,0.936
50+,686,0.017,0.922



By March traffic volume


,n_pages,base_rate,model_auc
band,,,
500-1k,16147,0.004,0.725
1k-5k,31511,0.043,0.796
5k-20k,11227,0.304,0.851
20k+,2057,0.471,0.918



Per-client AUC across 22 clients with >= 50 pages: worst 0.578 | median 0.931 | best 0.990
-> ML-06 signal 1 measured a 10x CTR spread between clients; this is that spread showing up as unevenness in where the model can be trusted.


In [54]:
# --- Three concrete wrong cases, and why each is hard ------------------------
top50 = df.nsmallest(50, "model_rank")
n_wrong_top50 = int((top50.label == 0).sum())
print(f"of the model's top 50, {n_wrong_top50} did NOT persist into April")

if n_wrong_top50 >= 3:
    wrong = top50[top50.label == 0].nsmallest(3, "model_rank")
    print("(taken from the top 50)\n")
else:
    wrong = df[df.label == 0].nlargest(3, "p")
    print(f"-> too few to inspect at K=50, so these are the model's most CONFIDENT mistakes "
          f"anywhere in the ranking instead. A queue with no errors in its top 50 is not a "
          f"perfect queue; it is a signal to look further down.\n")

for i, (_, r) in enumerate(wrong.iterrows(), 1):
    print(f"WRONG CASE {i} -- model rank #{int(r.model_rank)}, p={r.p:.2f}, label 0")
    print(f"  March: {r.impressions:,.0f} impressions at position {r.avg_position:.1f}, "
          f"CTR {r.mar_ctr_pp:.2f}% vs peers {r.mar_peer_pp:.2f}% (gap {r.mar_gap_pp:.2f}pp)")
    print(f"  April: {r.apr_impressions:,.0f} impressions at position {r.apr_position:.1f}, "
          f"gap {r.apr_gap_pp:.2f}pp -> below the flagging threshold")
    why = ("the page moved more than 2 positions between the months, so April compared it against "
           "a different peer group entirely" if abs(r.apr_position - r.avg_position) > 2 else
           "April traffic fell by more than half, so the gap shrank because the page got quieter, "
           "not because it got better" if r.apr_impressions < 0.5 * r.impressions else
           "nothing visible changed -- the March gap sat inside the month-to-month noise this page "
           "normally shows, which is exactly the case a monthly snapshot cannot separate")
    print(f"  why it is hard: {why}\n")

of the model's top 50, 0 did NOT persist into April
-> too few to inspect at K=50, so these are the model's most CONFIDENT mistakes anywhere in the ranking instead. A queue with no errors in its top 50 is not a perfect queue; it is a signal to look further down.

WRONG CASE 1 -- model rank #140, p=0.93, label 0
  March: 21,225 impressions at position 7.2, CTR 0.04% vs peers 0.34% (gap 0.30pp)
  April: 4,733 impressions at position 6.9, gap 0.18pp -> below the flagging threshold
  why it is hard: April traffic fell by more than half, so the gap shrank because the page got quieter, not because it got better

WRONG CASE 2 -- model rank #144, p=0.93, label 0
  March: 43,135 impressions at position 3.2, CTR 0.06% vs peers 0.34% (gap 0.28pp)
  April: 2,330 impressions at position 4.1, gap -0.01pp -> below the flagging threshold
  why it is hard: April traffic fell by more than half, so the gap shrank because the page got quieter, not because it got better

WRONG CASE 3 -- model rank #166, p=

### What the run showed — read this against the table above

**On the comparison.** The rule I set before looking: the model wins only if it beats the ML-07
rule at precision@50 *and* the bootstrap interval on the difference excludes zero. Both held —
1.00 vs 0.90 at K=50, and +0.107 at K=200 with a 95% interval of +0.055 to +0.158. So the model
beat the baseline, and the margin is not resampling noise.

**But a near-perfect score is a reason to investigate, not to celebrate.** Precision@50 of 1.00
on a 9.5% base rate is exactly the shape that should make an ML engineer suspicious. It is not
temporal leakage — the timeline is clean, features end 03-31 and outcomes start 04-01. It is the
label's construction: requiring April missed clicks ≥ 10 makes page size a near-prerequisite for
being flagged again, which is why the base rate runs from 0.4% in the 500–1k band to 47.1% at 20k+. But size alone
reaches only 0.42 at precision@50 against the model's 1.00, so the label being easy does not
make the model trivial — the top of the queue comes from combining size with the March gap,
not from either alone. The model's win is real; the
label is easier than it looks.

**On where it is wrong.** Two patterns to expect, both predicted by earlier notebooks: the
**position 1–2 band** (ML-06 proved the peer comparison is unfair there, so both the label and the
score are built on a shaky comparison) and **between clients** (ML-06 signal 1's 10x spread means
"normal" differs per client, and one global model has to average over that).

**The three wrong cases** are the useful part. Each is a page the model was confident about that did
not persist — and the reason is usually that the page *moved*, or its traffic dried up, rather than
that anyone improved its snippet. Which is the honest limit of this whole lane: I observe that a
gap closed, never why.

### Limitations, stated plainly

1. **Survivor selection, disclosed.** Pages with no readable April data were dropped. That uses
   outcome-window information to define the population, and the count is printed in §2. Pages that
   disappear entirely may be exactly the interesting ones.
2. **The label inherits the proxy's assumptions.** It is "would my ML-07 rule flag it again", so it
   carries the same tier thresholds and the same position-peer comparison ML-06 showed fails at
   positions 1–2.
3. **One month predicting one month**, 36 clients, one season. Nothing here establishes that the
   pattern holds in May, and June stays sealed.
4. **No causal claim, unchanged since ML-02.** Persistence is not causation: this ranks pages whose
   problem looks durable, and says nothing about whether editing them helps.
5. **The label cannot separate "fixed" from "died."** Two of the model's three most confident
   mistakes are pages whose April traffic fell by more than half — their gap stopped clearing the
   missed-clicks gate because demand left, not because the page improved. A queue built on this
   label will treat a dying page as a solved page.

In [55]:
# --- Receipts ---------------------------------------------------------------
import json, os
os.makedirs("work/outputs", exist_ok=True)

receipts = {
    "assignment": "ML-08 - Capstone Modeling Lane",
    "design": {"features_month": FEATURE_MONTH, "outcome_month": OUTCOME_MONTH,
               "decision_moment": DECISION_MOMENT,
               "label": "would the ML-07 rule flag this page again in April "
                        f"(gap >= {GAP_MIN}pp and missed clicks >= {CLICKS_MIN})",
               "label_type": "OBSERVED outcome (first in this project), not an authored proxy",
               "validation": "GroupKFold(5) on client_hash_id, all metrics out-of-fold",
               "seed": SEED},
    "population": {"march_eligible": int(n_before), "with_april_outcome": int(len(df)),
                   "dropped_no_april_outcome": int(n_before - len(df)),
                   "clients": int(df.client_hash_id.nunique()),
                   "base_rate": round(float(base), 4)},
    "comparison": table.to_dict(orient="records"),
    "bootstrap_diff_at_200": {"mean": round(float(np.mean(boot)), 4),
                              "ci95": [round(float(lo), 4), round(float(hi), 4)],
                              "resamples": 500},
    "split_gap": {"auc_random_split": round(float(auc_random), 3),
                  "auc_grouped_split": round(float(auc_grouped), 3)},
    "top_features": imp.head(5).to_dict(orient="records"),
    "claim_discipline": "observed persistence, measured out-of-fold, decision-support only; "
                        "no causal claim about editing pages",
}
with open("work/outputs/ml08_model_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2, default=float)

print(table.to_string(index=False))
print(f"\nbase rate {base:.3f} | rows {len(df):,} | clients {df.client_hash_id.nunique()}")
print("\nsaved -> work/outputs/ml08_model_receipts.json")

                           model  precision@50  precision@200  ROC AUC  lift@50
      random ranking (base rate)          0.12          0.085    0.494     1.26
ML-07 rule (March missed clicks)          0.90          0.875    0.864     9.48
   LogReg  - shape features only          0.48          0.385    0.878     5.06
   RF      - shape features only          0.90          0.845    0.881     9.48
     RF      - shape + March gap          1.00          0.980    0.922    10.53

base rate 0.095 | rows 60,942 | clients 34

saved -> work/outputs/ml08_model_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What this hands forward

1. **The project finally has an observed label.** Everything through ML-07 ranked by a rule I wrote;
   this ranks by something April actually did. The capstone should lead with that distinction.
2. **The baseline stands or falls on evidence, not on hope.** Whichever way §3's table fell, it was
   computed on the same rows, the same split and the same metric — and a tie is reported as a tie.
3. **Two known-weak regions carry into the paper**: positions 1–2, where ML-06 showed the peer
   comparison is unfair, and between-client variation, which per-client normalisation should be the
   next experiment against.